# Defenses: closing the leak

The leak exists because the commitment ψ depends on the witness φ.
We show the two defenses and measure the **φ-dependence** of ψ —
the artifact-free quantity (the paper's own leakage metric):
`max spread of P[ψ(i)=v | φ] across different φ`.

- **Simulator-aligned** (§4.1): train ψ to uniform-on-remaining ⇒ ψ
  becomes uniform and φ-independent.
- **Witness-masking** (§4.2): zero the φ block at inference ⇒ ψ
  becomes φ-independent (though not necessarily uniform).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root
import torch, itertools, math
torch.manual_seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


device: cuda


In [2]:
from subliminal.layout import perm_layout
from subliminal.data import make_perm_dataset, build_perm_sequences, rand_perms
from subliminal.train import train_prover
from subliminal.contexts import PermContext
from subliminal.sample import sample_psi, marginal_matrix
n = 5; layout = perm_layout(n)
tr = build_perm_sequences(*make_perm_dataset(3000, n, 0))
vl = build_perm_sequences(*make_perm_dataset(500, n, 1))
base = train_prover(layout, {'psi':'ce','psi_inv':'ce','phi_psi_inv':'ce'},
    tr, vl, steps=4000, batch=32, lr=3e-4, seed=0, ckpt_path='/tmp/def_base.pt',
    eval_every=10**9, log_every=10**9)
simal = train_prover(layout, {'psi':'uniform','psi_inv':'ce','phi_psi_inv':'ce'},
    tr, vl, steps=4000, batch=32, lr=3e-4, seed=0, ckpt_path='/tmp/def_sa.pt',
    eval_every=10**9, log_every=10**9)


/home/akash10/miniconda3/envs/aug-spm/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  step      0  psi=2.4255  psi_inv=2.3844  phi_psi_inv=2.2052


  step   3999  psi=0.8448  psi_inv=0.0001  phi_psi_inv=0.0004


  saved /tmp/def_base.pt


/home/akash10/miniconda3/envs/aug-spm/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  step      0  psi=2.3981  psi_inv=2.3844  phi_psi_inv=2.2052


  step   3999  psi=0.9603  psi_inv=0.0000  phi_psi_inv=0.0272


  saved /tmp/def_sa.pt


### Measure φ-dependence (the leak channel) and non-uniformity of ψ


In [3]:
def stats(model, zero_blocks=()):
    g = torch.Generator().manual_seed(1)
    Ms = []
    for p in rand_perms(6, n, g):
        ctx = p.unsqueeze(0).repeat(4000, 1)
        Ms.append(marginal_matrix(sample_psi(model, ctx, layout, valid=True,
                                             zero_blocks=zero_blocks), n))
    Ms = torch.stack(Ms)
    phidep = (Ms.max(0).values - Ms.min(0).values).abs().max().item()
    nonunif = (Ms - 1.0/n).abs().max().item()
    return phidep, nonunif
print('model            phi-dependence   non-uniformity')
print('baseline (leak)  %.3f            %.3f' % stats(base))
print('simulator-aligned %.3f            %.3f' % stats(simal))
print('witness-masked   %.3f            %.3f' % stats(base, (layout['phi'],)))


model            phi-dependence   non-uniformity


baseline (leak)  0.417            0.291


simulator-aligned 0.027            0.031


witness-masked   0.032            0.164


Both defenses drive **φ-dependence to the sampling-noise floor**
(~0.02), closing the leak. Simulator-aligned additionally makes ψ
uniform; witness-masking leaves ψ biased but φ-independent — which is
all zero-knowledge requires.
